In [ ]:
import pandas as pd
from analysis_utils import filter_data, get_simulator, generate_session_results, TypeScenario
from pathlib import Path

%load_ext autoreload
%autoreload 2

import datetime
import os

import warnings
warnings.filterwarnings("ignore")

Select your parameters below

In [2]:
months = range(1, 2)
year = 2023
scenario: TypeScenario = "threshold"

Run the simulation

In [ ]:
sessions_file = Path(__name__).resolve().parents[1] / "data" / "Sessions3.csv"
sessions_df = pd.read_csv(sessions_file)
sessions_df = sessions_df.sort_values(by="startChargeTime")

# Create output folder for this simulation
current_time = datetime.datetime.now()
time_str = current_time.strftime("%Y-%m-%d_%H-%M-%S")
folder_path = f"results/{scenario}/{time_str}"
os.makedirs(folder_path, exist_ok=True)

for month in months:
    # function to filter data in the same way for each scenario
    test_df = filter_data(sessions_df, month, year, "all_scheduled")

    # function to run a scenario (returns a child of BaselineSimulator)
    sim = get_simulator(test_df, scenario)

    # Run simulation and save results and append to the summary
    results_file_name = f"{folder_path}/{month}_{year}_{scenario}.csv"
    summary_file_name = f"{folder_path}/summary.csv"
    session_results = generate_session_results(
        sim, month, results_file_name, summary_file_name
    )

# Tests peak prediction

In [12]:
import numpy as np
import json
import pandas as pd
%load_ext autoreload
%autoreload 2
from peak_forecast_simulator import PeakForecastSimulator

In [37]:
PeakSimulator = PeakForecastSimulator(pd.DataFrame())

In [7]:
with open("samples_features_peak_pred.json", "r") as f:
    feature_samples = json.load(f)

In [44]:
def reverse_normalize(PeakSimulator, features):
    reversed_prediction = (
        features
        * (
            PeakSimulator.features_norm_parameters_max
            - PeakSimulator.features_norm_parameters_min
        )
        + PeakSimulator.features_norm_parameters_min
    )
    return reversed_prediction


for sample in feature_samples:
    reversed_sample = reverse_normalize(PeakSimulator, feature_samples[sample])
    feature_samples[sample] = reversed_sample

In [ ]:
sample = feature_samples["sample_13"]
prediction = PeakSimulator.make_prediction(sample, workday=1)
PeakSimulator.visualize_samples(sample, prediction)